# 🇲🇦 Pipeline de Data Science pour la Prévision Touristique et de ROI Hôtelier (Coupe du Monde FIFA 2030)

Ce notebook compile l'intégralité du pipeline de traitement des données, d'analyse exploratoire, d'ingénierie des caractéristiques, de modélisation prédictive multi-cible (Arrivées et Nuitées) et de simulation financière de ROI hôtelier à l'horizon 2035.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath('..'))
from src.config import DATA_DIR, FIGURES_DIR, TARGET_COL, NIGHTS_COL, TRAIN_END_DATE, TEST_START_DATE
import src.data_loader as loader
import src.cleaning as cleaner
import src.features as feat
import src.metrics as metrics_mod
from src.roi_simulator import HotelROISimulator

## 1. Chargement et Nettoyage des Données Multi-Sources
Nous fusionnons les indicateurs économiques mondiaux et nationaux avec les statistiques d'arrivées et de nuitées. Nous intégrons les données réelles du COVID-19 et reconstruisons les séries historiques par désagrégation temporelle saisonnière.

In [ ]:
print("Chargement et nettoyage du dataset...")
df = loader.load_and_merge_tourism_data()
df = cleaner.integrate_covid_data(df)
df = cleaner.reconstruct_historical_arrivals(df)
df = cleaner.reconstruct_historical_receipts(df)

print(f"Dimensions du dataset final : {df.shape}")
df.head(5)

## 2. Ingénierie des Caractéristiques (Feature Engineering)
Nous construisons 49 variables prédictives : lags, statistiques mobiles, encodages cycliques pour la saisonnalité, jours fériés nationaux, calendrier lunaire pour le Ramadan, anomalies climatiques et géopolitiques, et variables macroéconomiques retardées.

In [ ]:
print("Génération des caractéristiques prédictives...")
df_featured = feat.build_features(df)

features_arrivals = feat.get_feature_list()
features_nights = feat.get_nights_feature_list()

print(f"Nombre de features générées pour les Arrivées : {len(features_arrivals)}")
print(f"Nombre de features générées pour les Nuitées : {len(features_nights)}")

## 3. Modélisation Prédictive Comparée
Nous comparons 13 modèles (SARIMA, Prophet, 9 modèles de Machine Learning classiques, LSTM et SimpleRNN) sur les deux cibles de prévision. Le classement des performances R², RMSE, MAE, MAPE est automatiquement calculé sur le jeu de test chronologique (2023-2026).

In [ ]:
# Lecture et affichage des classements sauvegardés lors de l'évaluation globale
df_metrics_arrivals = pd.read_csv(os.path.join(DATA_DIR, 'model_performance_metrics.csv'))
df_metrics_nights = pd.read_csv(os.path.join(DATA_DIR, 'model_performance_metrics_nuitees.csv'))
df_top_3_overall = pd.read_csv(os.path.join(DATA_DIR, 'top_3_models_overall.csv'))

print("=== CLASSEMENT DES MODÈLES POUR LES ARRIVÉES TOURISTIQUES ===")
display(df_metrics_arrivals.head(5))

print("\n=== CLASSEMENT DES MODÈLES POUR LES NUITÉES NATIONALES ===")
display(df_metrics_nights.head(5))

print("\n=== MEILLEURS MODÈLES RETENUS PAR CIBLE (TOP 3) ===")
display(df_top_3_overall)

## 4. Visualisation des Prédictions et Projections 2035
Nous visualisons les courbes réelles versus prédites sur l'ensemble de test, ainsi que les tendances de croissance prévues à l'horizon 2035 avec le boost exceptionnel de la Coupe du Monde FIFA 2030.

In [ ]:
# Affichage des graphes de comparaison générés par le script
fig_arr_path = os.path.join(FIGURES_DIR, '05_arrivals_test_comparison_top3.png')
fig_ngt_path = os.path.join(FIGURES_DIR, '09_nights_test_comparison_top3.png')

if os.path.exists(fig_arr_path):
    img_arr = plt.imread(fig_arr_path)
    plt.figure(figsize=(12, 6))
    plt.imshow(img_arr)
    plt.axis('off')
    plt.show()
    
if os.path.exists(fig_ngt_path):
    img_ngt = plt.imread(fig_ngt_path)
    plt.figure(figsize=(12, 6))
    plt.imshow(img_ngt)
    plt.axis('off')
    plt.show()

## 5. Simulation Financière Hôtelière Comparée et Rentabilité
Nous comparons l'approche par Arrivées et l'approche par Nuitées dans l'évaluation de la rentabilité sur 10 ans d'un hôtel de 200 chambres à Marrakech (WACC = 8%). L'occupation est soit déduite par le ratio de croissance (Arrivées), soit calculée directement et de façon plus robuste à partir des nuitées réelles projetées.

In [ ]:
# Instanciation du simulateur
sim = HotelROISimulator(
    rooms=200,
    investment_usd=40000000.0,
    opex_margin=0.65,
    discount_rate=0.08,
    base_occupancy=0.68,
    base_adr=250.0,
    wc_adr_boost_pct=0.40,
    inflation_rate=0.025
)

# Simulation classique sur 10 ans avec boost statique
df_roi = sim.simulate_10years()
metrics = sim.calculate_metrics(df_roi)

print(f"=== ROI HÔTELIER CLASSIQUE : SCÉNARIO COUPE DU MONDE 2030 ===")
print(f"NPV (Valeur Actuelle Nette) : {metrics['NPV_WC_MUSD']:.2f} Millions USD")
print(f"IRR (Taux de Rentabilité)   : {metrics['IRR_WC_Pct']:.2f}%")
print(f"Payback (Retour Invest.)     : {metrics['Payback_WC_Years']} ans")
print(f"ROI Cumulé sur 10 ans       : {metrics['ROI_WC_Pct']:.2f}%")